# Resume Variants Dataset Creation

### Load CSV files

In [ ]:
import pandas as pd
import re
import random
import io
from google.colab import files
from IPython.display import display

print('Please upload your two CSV files:')
uploaded = files.upload()

filenames    = list(uploaded.keys())
resumes_file = next(f for f in filenames if 'base' in f or 'resume' in f.lower())
unis_file    = next(f for f in filenames if 'unis' in f.lower() or 'dataset' in f.lower())

df_resumes = pd.read_csv(io.BytesIO(uploaded[resumes_file]))
df_lists   = pd.read_csv(io.BytesIO(uploaded[unis_file]))

print(f'✓ Resumes loaded : {len(df_resumes)} rows')
print(f'✓ Lists loaded   : {len(df_lists)} rows')

Please upload your two CSV files:


Saving unis_companies_dataset.csv to unis_companies_dataset.csv
Saving base_resumes.csv to base_resumes.csv
✓ Resumes loaded : 30 rows
✓ Lists loaded   : 40 rows


### Preview Raw Data

In [ ]:
print('=== Base Resumes ===')
display(df_resumes.head(3))
display(df_resumes.tail(3))

print('\n=== Universities & Companies List ===')
display(df_lists.head(3))
display(df_lists.tail(3))

=== Base Resumes ===


,Category,Quantification,Gender,Ethnicity,Resume
0,Software Engineer,Highly quantified,Female,American Indian,Name: Aiyana Morningstar; Skills - Front-end S...
1,Software Engineer,Low quantified,Female,Asian,Name: Li Mei Chen; Experience 1 - Company: Com...
2,Software Engineer,Low quantified,Female,Black,Name: Aaliyah Thompson; Skills - Proficient: A...


,Category,Quantification,Gender,Ethnicity,Resume
27,Product Manager,Highly quantified,Male,Black,"Name: Kofi Asante; Summary: Customer-focused, ..."
28,Product Manager,Highly quantified,Male,Pacific Islander,Name: Tane Faleolo; Skills - Product: Roadmapp...
29,Product Manager,Highly quantified,Male,White,Name: Liam O'Brien; Summary: Product Professio...



=== Universities & Companies List ===


,Name,Type,Credentials
0,Carnegie Mellon University,University,Elite
1,Univ. of Illinois at Urbana-Champaign,University,Elite
2,Univ. of California - San Diego,University,Elite


,Name,Type,Credentials
37,ScaleUp,Company,Non-elite
38,LaunchPad,Company,Non-elite
39,CodeNest,Company,Non-elite


## PHASE 1: GENERATE CREDENTIAL VARIANTS

### Build Credential Lookup List

In [ ]:
elite_unis    = df_lists[(df_lists['Type'] == 'University') & (df_lists['Credentials'] == 'Elite')   ]['Name'].str.strip().tolist()
elite_cos     = df_lists[(df_lists['Type'] == 'Company')    & (df_lists['Credentials'] == 'Elite')   ]['Name'].str.strip().tolist()
nonelite_unis = df_lists[(df_lists['Type'] == 'University') & (df_lists['Credentials'] == 'Non-elite')]['Name'].str.strip().tolist()
nonelite_cos  = df_lists[(df_lists['Type'] == 'Company')    & (df_lists['Credentials'] == 'Non-elite')]['Name'].str.strip().tolist()

summary = pd.DataFrame({
    'Category'       : ['Universities', 'Companies'],
    'Elite'          : [elite_unis,     elite_cos],
    'Non-Elite'      : [nonelite_unis,  nonelite_cos],
})
display(summary)

,Category,Elite,Non-Elite
0,Universities,"[Carnegie Mellon University, Univ. of Illinois...","[University of Utah, Kansas State University, ..."
1,Companies,"[Meta, Netflix, Apple, Google, Amazon, Microso...","[StartupX, TechStartup, GrowthCo, InsightAI, B..."


### Define Replacement Functions

In [ ]:
def replace_institutions(text, unis_pool, seed=None):
    rng = random.Random(seed)
    pool = unis_pool.copy()
    rng.shuffle(pool)
    pool_cycle = pool * 10
    counter = [0]
    def replacer(match):
        replacement = pool_cycle[counter[0] % len(pool)]
        counter[0] += 1
        return f'Institution: {replacement}'
    return re.sub(r'Institution:\s*[^;]+?(?=;|$)', replacer, text)

def replace_companies(text, cos_pool, seed=None):
    rng = random.Random(seed)
    pool = cos_pool.copy()
    rng.shuffle(pool)
    pool_cycle = pool * 10
    counter = [0]
    def replacer(match):
        replacement = pool_cycle[counter[0] % len(pool)]
        counter[0] += 1
        return f'Company: {replacement}'
    return re.sub(r'Company:\s*[^;]+?(?=;|$)', replacer, text)

def make_resume_variant(text, unis_pool, cos_pool, seed=None):
    text = replace_institutions(text, unis_pool, seed=seed)
    text = replace_companies(text, cos_pool, seed=seed)
    return text

print('✓ Replacement functions defined')

✓ Replacement functions defined


### Apply Credential Replacements

In [ ]:
df_resumes['Resume_Elite'] = [
    make_resume_variant(row['Resume'], elite_unis, elite_cos, seed=idx)
    for idx, row in df_resumes.iterrows()
]

df_resumes['Resume_NonElite'] = [
    make_resume_variant(row['Resume'], nonelite_unis, nonelite_cos, seed=idx)
    for idx, row in df_resumes.iterrows()
]

print(f'✓ Generated Resume_Elite and Resume_NonElite for {len(df_resumes)} resumes')

✓ Generated Resume_Elite and Resume_NonElite for 30 resumes


### Credential Sanity Check

In [ ]:
sample_idx = 0  # ← change to check a different resume

def extract(text, field):
    pattern = rf'{field}:\s*([^;]+?)(?:;|$)'
    return re.findall(pattern, text)

rows = []
for field, label in [('Institution', 'University'), ('Company', 'Company')]:
    orig = extract(df_resumes.loc[sample_idx, 'Resume'],          field)
    eli  = extract(df_resumes.loc[sample_idx, 'Resume_Elite'],    field)
    ne   = extract(df_resumes.loc[sample_idx, 'Resume_NonElite'], field)
    for i, (o, e, n) in enumerate(zip(orig, eli, ne)):
        rows.append({'Field': f'{label} {i+1}', 'Original': o.strip(),
                     'Elite': e.strip(), 'Non-Elite': n.strip()})

sanity_df = pd.DataFrame(rows)
print(f'=== Sanity Check: Resume #{sample_idx} ===')
display(sanity_df)

=== Sanity Check: Resume #0 ===


,Field,Original,Elite,Non-Elite
0,University 1,WelTec Petone,Cornell University,Stockton University
1,University 2,Chapman University,University of Washington,University of Nebraska -- Lincoln
2,Company 1,Numeral Studio,NVIDIA,ScaleUp
3,Company 2,Habitual Genesis,Palantir,LaunchPad
4,Company 3,PartsTrader Markets Ltd,Netflix,TechStartup
5,Company 4,WorkTango (formerly KazooHR),Microsoft,OrbitX
6,Company 5,Conde Nast,Google,InsightAI
7,Company 6,Bypass Mobile,Amazon,Buildr
8,Company 7,Union Metrics,Apple,GrowthCo
9,Company 8,The Flatiron School,Meta,StartupX


### Save and Download

In [ ]:
output_filename = 'resumes_credential_variants.csv'
df_resumes[['Category', 'Quantification', 'Gender', 'Ethnicity',
            'Resume', 'Resume_Elite', 'Resume_NonElite']].to_csv(output_filename, index=False)

print(f'✓ Saved to {output_filename}')
files.download(output_filename)

✓ Saved to resumes_credential_variants.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## PHASE 2: GENERATE QUANTIFICATION VARIANTS

### INSTALL & IMPORTS

In [ ]:
# Uses Hugging Face Inference API
import pandas as pd
import re
import random
import io
from google.colab import files
from IPython.display import display
!pip install -q huggingface_hub

### Load CSV

In [ ]:
print("Upload your credential-variants CSV (resumes_credential_variants.csv):")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname]))

print(f"\nLoaded {len(df)} rows")
print("Columns:", df.columns.tolist())
print("Quantification values:", df['Quantification'].unique())

Upload your credential-variants CSV (resumes_credential_variants.csv):


Saving resumes_credential_variants - resumes_credential_variants.csv to resumes_credential_variants - resumes_credential_variants (1).csv

Loaded 30 rows
Columns: ['Category', 'Quantification', 'Gender', 'Ethnicity', 'Resume', 'Resume_Elite', 'Resume_NonElite']
Quantification values: ['Highly quantified' 'Low quantified']


### API CONFIG

In [ ]:
import getpass
import time
from huggingface_hub import InferenceClient
HF_TOKEN = getpass.getpass("Enter your Hugging Face API token (won't be shown): ")

client = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=HF_TOKEN,
)

Enter your Hugging Face API token (won't be shown): ··········


### PROMPTS

In [ ]:
STRIP_PROMPT = """You are a professional resume editor. Your task is to REMOVE quantified metrics from the Responsibilities sections of this resume to create a MINIMAL QUANTIFICATION version.

Rules:
- Only edit text inside "Responsibilities:" sections.
- Replace specific numbers, percentages, dollar amounts, and counts with vague qualitative language.
  Examples:
    "increased user engagement by 35%" → "increased user engagement"
    "reduced load time by 20-30%"      → "improved load time"
    "580+ commits"                      → "numerous commits"
    "100 most common customer issues"  → "common customer issues"
    "90% graduation rate"              → "strong graduation rate"
- Do NOT change company names, university names, job titles, dates, technologies, skills, or any other section.
- Return ONLY the full modified resume text. No explanation, no preamble.

RESUME:
{resume}"""

ENHANCE_PROMPT = """You are a professional resume editor. Your task is to ADD quantified metrics to the Responsibilities sections of this resume to create a HIGH QUANTIFICATION version.

Rules:
- Only edit text inside "Responsibilities:" sections.
- Add plausible, realistic numbers, percentages, and measurable outcomes to vague statements.
  Examples:
    "improved application performance"          → "improved application performance by 35%"
    "worked with a team to build applications"  → "collaborated with a 6-person team to build 3 cross-platform applications"
    "wrote code for scheduling posts"           → "wrote optimized Django code reducing scheduling latency by 40%"
- Make numbers realistic and consistent with the seniority level and role described.
- Do NOT change company names, university names, job titles, dates, technologies, skills, or any other section.
- Return ONLY the full modified resume text. No explanation, no preamble.

RESUME:
{resume}"""

### LLM CALL FUNCTION

In [ ]:
def call_llm(prompt, max_new_tokens=4096, retries=3, wait=10):
    """Call Mistral-7B via HF chat_completion API with retry logic."""
    for attempt in range(retries):
        try:
            response = client.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_new_tokens,
                temperature=0.3,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  ⚠️  Attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
    print("  ❌ All retries failed. Returning original text.")
    return None

### DETERMINE WHICH PROMPT TO USE

In [ ]:
# Logic:
#   "Highly quantified" base resume → STRIP  to get minimal version
#   "Low quantified"    base resume → ENHANCE to get high version

def get_prompt_and_target(row, col):
    """
    For a given resume column, return (prompt_template, description).
    We apply the OPPOSITE transformation to the base quantification level.
    """
    quant = row['Quantification'].strip().lower()
    if 'high' in quant:
        # Base is already high-quant → strip it down
        return STRIP_PROMPT, 'min_quant'
    else:
        # Base is low-quant → enhance it
        return ENHANCE_PROMPT, 'high_quant'

### GENERATE VARIANTS

In [ ]:
# For each of the 3 resume columns (Resume, Resume_Elite, Resume_NonElite),
# generate the opposite-quantification version.
# Total calls: 30 resumes × 3 columns = 90 API calls

TARGET_COLS = ['Resume', 'Resume_Elite', 'Resume_NonElite']
OUTPUT_COLS = {
    'Resume'          : ('Resume_HighQuant',       'Resume_MinQuant'),
    'Resume_Elite'    : ('Resume_Elite_HighQuant',  'Resume_Elite_MinQuant'),
    'Resume_NonElite' : ('Resume_NonElite_HighQuant','Resume_NonElite_MinQuant'),
}

# Initialise output columns with originals as fallback
for col, (high_col, min_col) in OUTPUT_COLS.items():
    df[high_col] = df[col]
    df[min_col]  = df[col]

total_calls = len(df) * len(TARGET_COLS)
call_count  = 0

for col in TARGET_COLS:
    high_col, min_col = OUTPUT_COLS[col]
    print(f"\n{'='*60}")
    print(f"Processing column: {col}")
    print(f"{'='*60}")

    for idx, row in df.iterrows():
        call_count += 1
        quant = row['Quantification'].strip().lower()

        print(f"  [{call_count}/{total_calls}] Row {idx} | {row['Quantification']} | {col}")

        if 'high' in quant:
            # Base is highly quantified → generate a stripped (min-quant) version
            prompt   = STRIP_PROMPT.format(resume=row[col])
            result   = call_llm(prompt)
            if result:
                df.at[idx, min_col]  = result
                df.at[idx, high_col] = row[col]   # original is already high
        else:
            # Base is low quantified → generate an enhanced (high-quant) version
            prompt   = ENHANCE_PROMPT.format(resume=row[col])
            result   = call_llm(prompt)
            if result:
                df.at[idx, high_col] = result
                df.at[idx, min_col]  = row[col]   # original is already min

        time.sleep(1)   # gentle rate limiting

print("\n✅ All quantification variants generated!")


Processing column: Resume
  [1/90] Row 0 | Highly quantified | Resume
  [2/90] Row 1 | Low quantified | Resume
  [3/90] Row 2 | Low quantified | Resume
  [4/90] Row 3 | Highly quantified | Resume
  [5/90] Row 4 | Highly quantified | Resume
  [6/90] Row 5 | Highly quantified | Resume
  [7/90] Row 6 | Highly quantified | Resume
  [8/90] Row 7 | Highly quantified | Resume
  [9/90] Row 8 | Highly quantified | Resume
  [10/90] Row 9 | Low quantified | Resume
  [11/90] Row 10 | Highly quantified | Resume
  [12/90] Row 11 | Highly quantified | Resume
  [13/90] Row 12 | Low quantified | Resume
  [14/90] Row 13 | Highly quantified | Resume
  [15/90] Row 14 | Highly quantified | Resume
  [16/90] Row 15 | Low quantified | Resume
  [17/90] Row 16 | Highly quantified | Resume
  [18/90] Row 17 | Highly quantified | Resume
  [19/90] Row 18 | Highly quantified | Resume
  [20/90] Row 19 | Highly quantified | Resume
  [21/90] Row 20 | Highly quantified | Resume
  [22/90] Row 21 | Highly quantified | Re

### RESHAPE TO FINAL 4-VARIANT FORMAT

In [ ]:
# Group A: Elite   + High Quant   (n=30)
# Group B: Elite   + Min  Quant   (n=30)
# Group C: NonElite + High Quant  (n=30)
# Group D: NonElite + Min  Quant  (n=30)

rows = []
for _, row in df.iterrows():
    base = {
        'Category'            : row['Category'],
        'Gender'              : row['Gender'],
        'Ethnicity'           : row['Ethnicity'],
        'Base_Quantification' : row['Quantification'],
    }

    # Group A
    rows.append({**base,
        'Group'          : 'A',
        'Credentials'    : 'Elite',
        'Quantification' : 'High',
        'Resume'         : row['Resume_Elite_HighQuant'],
    })
    # Group B
    rows.append({**base,
        'Group'          : 'B',
        'Credentials'    : 'Elite',
        'Quantification' : 'Minimal',
        'Resume'         : row['Resume_Elite_MinQuant'],
    })
    # Group C
    rows.append({**base,
        'Group'          : 'C',
        'Credentials'    : 'NonElite',
        'Quantification' : 'High',
        'Resume'         : row['Resume_NonElite_HighQuant'],
    })
    # Group D
    rows.append({**base,
        'Group'          : 'D',
        'Credentials'    : 'NonElite',
        'Quantification' : 'Minimal',
        'Resume'         : row['Resume_NonElite_MinQuant'],
    })

df_final = pd.DataFrame(rows)

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Groups: {df_final['Group'].value_counts().to_dict()}")


Final dataset shape: (120, 8)
Groups: {'A': 30, 'B': 30, 'C': 30, 'D': 30}


### SANITY CHECK

In [ ]:
sample_idx = 0
sample_base = df.iloc[sample_idx]

print(f"\n=== Sanity Check: Row {sample_idx} | {sample_base['Category']} | {sample_base['Quantification']} ===")

# Show first Responsibilities block from each of the 4 variants
for group, col in [('A - Elite High', 'Resume_Elite_HighQuant'),
                   ('B - Elite Min',  'Resume_Elite_MinQuant'),
                   ('C - NonElite High', 'Resume_NonElite_HighQuant'),
                   ('D - NonElite Min',  'Resume_NonElite_MinQuant')]:

    text  = sample_base[col]
    match = re.search(r'Responsibilities:\s*([^;]{80,})', text)
    snippet = match.group(1)[:250] if match else text[:250]
    print(f"\n--- Group {group} ---")
    print(snippet + "...")


=== Sanity Check: Row 0 | Software Engineer | Highly quantified ===

--- Group A - Elite High ---
Delivered multi-client throughput via 580+ commits with consistently green lint/type/test gates by pairing tightly with design, PM, and back-end partners across compliance, healthcare, marketing, and conference products...

--- Group B - Elite Min ---
Delivered multi-client throughput via numerous commits with consistently green lint/type/test gates by pairing tightly with design, PM, and back-end partners across compliance, healthcare, marketing, and conference products...

--- Group C - NonElite High ---
Delivered multi-client throughput via 580+ commits with consistently green lint/type/test gates by pairing tightly with design, PM, and back-end partners across compliance, healthcare, marketing, and conference products...

--- Group D - NonElite Min ---
Delivered multi-client throughput through numerous commits with consistently green lint/type/test gates by pairing tightly with design

### SAVE & DOWNLOAD

In [ ]:
wide_filename  = 'resumes_all_variants_wide.csv'
final_filename = 'resumes_final_4groups.csv'

df.to_csv(wide_filename, index=False)
df_final.to_csv(final_filename, index=False)

print(f"\n✅ Saved wide format  → {wide_filename}")
print(f"✅ Saved final groups → {final_filename}")

files.download(wide_filename)
files.download(final_filename)


✅ Saved wide format  → resumes_all_variants_wide.csv
✅ Saved final groups → resumes_final_4groups.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>